In [ ]:
pip install transformers datasets soundfile speechbrain accelerate

In [ ]:
import torch
import numpy as np
from datasets import load_dataset, Audio
from transformers import (
    SpeechT5Processor,
    SpeechT5ForTextToSpeech,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from speechbrain.inference import EncoderClassifier

In [ ]:
dataset = load_dataset("facebook/voxpopuli", "ro", split="train")

# Shuffle and subset (25%)
dataset = dataset.shuffle(seed=42)
dataset = dataset.select(range(len(dataset) // 4))

# Convert audio to 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading dataset shards:   0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
def get_duration(sample):
    return len(sample["audio"]["array"]) / 16000

durations = [get_duration(x) for x in dataset]

print("Min:", np.min(durations))
print("Max:", np.max(durations))
print("Avg:", np.mean(durations))

Min: 0.5199375
Max: 104.5399375
Avg: 11.484633183542215


In [ ]:
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")

model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")

speaker_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb",
    run_opts={"device": "cuda" if torch.cuda.is_available() else "cpu"}
)

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:spee

In [ ]:
def create_speaker_embedding(waveform):
    with torch.no_grad():
        waveform = torch.tensor(waveform).unsqueeze(0)
        embeddings = speaker_model.encode_batch(waveform)
        embeddings = torch.nn.functional.normalize(embeddings, dim=-1)
    return embeddings.squeeze().cpu().numpy()

In [ ]:
def normalize_text(text):
    replacements = {
        "â": "a", "ă": "a", "î": "i",
        "ș": "s", "ț": "t"
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text

dataset = dataset.map(lambda x: {
    "normalized_text": normalize_text(x["normalized_text"])
})

In [ ]:
from collections import Counter

speaker_counts = Counter(dataset["speaker_id"])

def filter_speakers(example):
    count = speaker_counts[example["speaker_id"]]
    return 120 <= count <= 350

dataset = dataset.filter(filter_speakers)

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]

    inputs = processor(
        text=example["normalized_text"],
        return_tensors="pt",
        truncation=True,
        max_length=600
    )

    labels = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    )

    label = labels["input_values"].squeeze(0)

    if label.dim() != 2:
        label = label.unsqueeze(-1)

    speaker_embedding = create_speaker_embedding(audio["array"])

    return {
        "input_ids": inputs["input_ids"].squeeze(0),          # tensor
        "labels": label,          # tensor (2D)
        "speaker_embeddings": torch.tensor(speaker_embedding) # tensor
    }

In [ ]:
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/1943 [00:00<?, ? examples/s]

In [ ]:
dataset = dataset.filter(lambda x: "input_ids" in x)
def filter_length(example):
    return len(example["input_ids"]) < 200

dataset = dataset.filter(filter_length)

In [ ]:
dataset.set_format(
    type="torch",
    columns=["input_ids", "labels", "speaker_embeddings"]
)

In [ ]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]
print(train_dataset[0])
print(type(train_dataset[0]["input_ids"]))

In [ ]:
from dataclasses import dataclass
import torch

@dataclass
class TTSDataCollator:

    def __call__(self, features):

        input_ids = [f["input_ids"] for f in features]
        labels = [f["labels"] for f in features]
        speaker_embeddings = [f["speaker_embeddings"] for f in features]

        # pad text
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=0
        )

        attention_mask = (input_ids != 0).long()

        # pad spectrograms
        max_len = max(l.shape[0] for l in labels)
        feature_dim = labels[0].shape[-1]

        padded_labels = []
        for l in labels:
            pad_len = max_len - l.shape[0]
            if pad_len > 0:
                pad = torch.full((pad_len, feature_dim), -100.0)
                l = torch.cat([l, pad], dim=0)
            padded_labels.append(l)

        labels = torch.stack(padded_labels)
        speaker_embeddings = torch.stack(speaker_embeddings)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "speaker_embeddings": speaker_embeddings
        }

data_collator = TTSDataCollator()

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="speecht5_finetuned_voxpopuli_ro",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=200,
    max_steps=1000,
    gradient_checkpointing=True,
    fp16=True,
    do_eval=True,
    per_device_eval_batch_size=4,
    save_steps=500,
    eval_steps=100,
    logging_steps=50,
    greater_is_better=False,
    label_names=["labels"],
    report_to=[]
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
from transformers import SpeechT5HifiGan

vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

example = eval_dataset[0]

speaker_embeddings = torch.tensor(example["speaker_embeddings"]).unsqueeze(0).to(device)

text = "salutare tuturor, acesta este un exemplu"

inputs = processor(text=text, return_tensors="pt").to(device)

speech = model.generate_speech(
    inputs["input_ids"],
    speaker_embeddings,
    vocoder=vocoder
)

In [ ]:
import soundfile as sf

sf.write("output.wav", speech.cpu().numpy(), samplerate=16000)